In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import json
import joblib
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

In [2]:
train = pd.read_csv("train_FE.csv")
test = pd.read_csv("test_FE.csv")

In [3]:
# Pisahkan fitur dan target
X = train.drop(columns=["id", "addicted_label"])
y = train["addicted_label"]

# Split internal: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [4]:
# Kolom kategorikal dan numerik
categorical_features = ["gender", "academic_work_impact"]

numeric_features = [
    col for col in X.columns
    if col not in categorical_features
]

# Preprocessing yang sama untuk seluruh model
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features
        )
    ]
)

# 10-fold CV
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [5]:
models = {

    "LightGBM": LGBMClassifier(
        objective="binary",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=2,
        verbosity=-1
    ),

    "XGBoost": XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        tree_method="hist",
        random_state=42,
        n_jobs=2
    ),

    "CatBoost": CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=500,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=3.0,
        random_seed=42,
        thread_count=2,
        verbose=0,
        allow_writing_files=False
    )
}

In [6]:
from sklearn.metrics import roc_auc_score
import joblib

results = []
trained_models = {}

for model_name, model in models.items():
    print(f"\nTraining: {model_name}")

    # Pipeline tetap sama, hanya model akhirnya berbeda
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # Training satu kali pada 80% data
    pipeline.fit(X_train, y_train)

    # Prediksi probabilitas pada 20% internal test
    y_test_prob = pipeline.predict_proba(X_test)[:, 1]

    # Hitung ROC-AUC internal test
    test_auc = roc_auc_score(y_test, y_test_prob)

    print(f"Internal Test ROC-AUC: {test_auc:.5f}")

    results.append({
        "model": model_name,
        "test_roc_auc": test_auc
    })

    # Simpan pipeline terlatih di RAM
    trained_models[model_name] = pipeline

    # Simpan file model agar tidak perlu fit ulang
    file_name = model_name.lower().replace(" ", "_") + "_baseline.joblib"
    joblib.dump(pipeline, file_name)

# Ranking berdasarkan ROC-AUC internal test
model_ranking = (
    pd.DataFrame(results)
    .sort_values("test_roc_auc", ascending=False)
    .reset_index(drop=True)
)

print("\n===== RANKING BASELINE MODEL =====")
display(model_ranking)


Training: LightGBM


c:\Users\danis\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Internal Test ROC-AUC: 0.96047

Training: XGBoost
Internal Test ROC-AUC: 0.95941

Training: CatBoost
Internal Test ROC-AUC: 0.95107

===== RANKING BASELINE MODEL =====


,model,test_roc_auc
0,LightGBM,0.960470
1,XGBoost,0.959410
2,CatBoost,0.951067


### ketemu model terbaik: lightgbm

In [13]:
import pandas as pd
import numpy as np
import json
import joblib

from lightgbm import LGBMClassifier
from scipy.stats import randint, uniform, loguniform
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV
)
from sklearn.metrics import roc_auc_score

In [14]:
# 1. Pisahkan fitur dan target
X = train.drop(columns=["id", "addicted_label"]).copy()
y = train["addicted_label"].copy()

In [15]:
# 2. Tentukan fitur kategorikal
categorical_features = ["gender", "academic_work_impact"]

# LightGBM membaca pandas category sebagai fitur kategorikal asli
for col in categorical_features:
    X[col] = X[col].astype("category")

In [16]:
# 3. Split internal 80:20
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 4. Skema 10-fold CV
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [17]:
# 5. Model dasar LightGBM
lgbm = LGBMClassifier(
    objective="binary",
    random_state=42,
    n_jobs=1,          # dua fold paralel diatur oleh RandomizedSearchCV
    verbosity=-1,
    subsample_freq=1
)

# 6. Ruang parameter yang akan diuji
param_distributions = {
    "n_estimators": randint(300, 1001),
    "learning_rate": loguniform(0.01, 0.15),
    "num_leaves": randint(15, 80),
    "max_depth": [-1, 4, 6, 8, 10],
    "min_child_samples": randint(10, 101),
    "subsample": uniform(0.6, 0.4),          # rentang 0.6–1.0
    "colsample_bytree": uniform(0.6, 0.4),   # rentang 0.6–1.0
    "reg_alpha": loguniform(0.0001, 10.0),
    "reg_lambda": loguniform(0.0001, 10.0)
}

# 7. Randomized Search: 12 kandidat × 10 fold = 120 fit
random_search_lgbm = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="roc_auc",
    cv=cv,
    n_jobs=2,            # aman untuk RAM 8 GB
    pre_dispatch=2,
    verbose=3,
    random_state=42,
    refit=True,          # otomatis fit model terbaik pada seluruh X_train
    return_train_score=False
)

In [18]:
# 8. Mulai tuning
random_search_lgbm.fit(X_train, y_train)

Fitting 10 folds for each of 12 candidates, totalling 120 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LGBMClassifie... verbosity=-1)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': <scipy.stats....001F2845B9370>, 'learning_rate': <scipy.stats....001F2E83168D0>, 'max_depth': [-1, 4, ...], 'min_child_samples': <scipy.stats....001F2807A8950>, ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",12
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",2
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",3
,"pre_dispatch pre_dispatch: int, or str, default='2*n_jobs'Controls the number of jobs that get dispatched during parallelexecution. Reducing this number can be useful to avoid anexplosion of memory consumption when more jobs get dispatchedthan CPUs can process. This parameter can be:- None, in which case all the jobs are immediately created and spawned. Use this for lightweight and fast-running jobs, to avoid delays due to on-demand spawning of the jobs- An int, giving the exact number of total jobs that are spawned- A str, giving an expression as a function of n_jobs, as in '2*n_jobs'",2
,"random_state random_state: int, RandomState ins

In [24]:
# 9. Hasil terbaik dari 10-fold CV
best_lgbm = random_search_lgbm.best_estimator_

print("\n===== PARAMETER TERBAIK LIGHTGBM =====")
print(random_search_lgbm.best_params_)

print(f"\nBest 10-Fold CV ROC-AUC: "
      f"{random_search_lgbm.best_score_:.6f}")


===== PARAMETER TERBAIK LIGHTGBM =====
{'colsample_bytree': np.float64(0.8827429375390468), 'learning_rate': np.float64(0.07200770311577641), 'max_depth': -1, 'min_child_samples': 14, 'n_estimators': 789, 'num_leaves': 55, 'reg_alpha': np.float64(3.7566296134528474), 'reg_lambda': np.float64(1.7790693962001112), 'subsample': np.float64(0.7797802696552814)}

Best 10-Fold CV ROC-AUC: 0.963533


In [22]:
# 10. Uji pada internal test 20%
y_test_prob = best_lgbm.predict_proba(X_test)[:, 1]
internal_auc = roc_auc_score(y_test, y_test_prob)

print(f"Internal Test ROC-AUC: {internal_auc:.6f}")

Internal Test ROC-AUC: 0.962919


In [25]:
# 11. Simpan semua hasil
results_lgbm = (
    pd.DataFrame(random_search_lgbm.cv_results_)
    .sort_values("rank_test_score")
)

results_lgbm.to_csv("lightgbm_random_search_results.csv", index=False)

with open("lightgbm_best_params.json", "w") as file:
    json.dump(random_search_lgbm.best_params_, file, indent=4)

joblib.dump(best_lgbm, "lightgbm_tuned.joblib")

print("\nFile tersimpan:")
print("- lightgbm_random_search_results.csv")
print("- lightgbm_best_params.json")
print("- lightgbm_tuned.joblib")


File tersimpan:
- lightgbm_random_search_results.csv
- lightgbm_best_params.json
- lightgbm_tuned.joblib
